# Notebook 02 — CO₂ Spectral Line Profiles

## Overview
Molecular absorption lines are not infinitely narrow delta functions — physical broadening processes give them characteristic shapes.

This notebook explores:
1. **Doppler (Gaussian) broadening** — due to molecular thermal motion
2. **Pressure (Lorentzian) broadening** — due to molecular collisions
3. **Voigt profile** — the physical convolution of both effects
4. How pressure and temperature control the line widths in the atmosphere

## Doppler Broadening
Thermal velocity distribution → Gaussian line shape:

$$\Delta\nu_D = \frac{\nu_0}{c}\sqrt{\frac{2k_BT\ln 2}{m}}$$

## Pressure Broadening
Collisions interrupt coherent oscillation → Lorentzian line shape:

$$\Delta\nu_L = \gamma_{\text{air}} \left(\frac{P}{P_{\text{ref}}}\right)^{n_{\text{air}}} \left(\frac{T_{\text{ref}}}{T}\right)^{n_{\text{air}}}$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
from spectroscopy import (
    doppler_hwhm,
    lorentz_hwhm,
    gaussian_profile,
    lorentz_profile,
    voigt_profile_func,
    absorption_cross_section,
    demo_line_profiles,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Line Width vs Temperature and Pressure

Let's explore how Doppler and Lorentzian widths vary through the atmosphere.

In [ ]:
nu0 = 6250.0   # cm⁻¹  (1.6 µm CO₂ band)

# --- Temperature dependence (Doppler) ---
temps = np.linspace(180, 320, 100)
dnu_D = np.array([doppler_hwhm(nu0, T) for T in temps])

# --- Pressure dependence (Lorentz) at T=250 K ---
pressures = np.linspace(1000, 101325, 100)
dnu_L = np.array([lorentz_hwhm(0.07, P, 250.0) for P in pressures])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(temps, dnu_D * 1000, color='royalblue', lw=2)
ax1.set_xlabel('Temperature [K]'); ax1.set_ylabel('Doppler HWHM [×10⁻³ cm⁻¹]')
ax1.set_title('Doppler Broadening vs Temperature  (∝ √T)')
ax1.grid(True, alpha=0.3)

ax2.plot(pressures / 100, dnu_L, color='tomato', lw=2)
ax2.set_xlabel('Pressure [hPa]'); ax2.set_ylabel('Lorentz HWHM [cm⁻¹]')
ax2.set_title('Pressure Broadening vs Pressure  (∝ P)')
ax2.grid(True, alpha=0.3)

plt.suptitle('Line Width Dependence on Atmospheric State', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/02a_linewidths.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Comparing Profile Shapes

Compare the Gaussian, Lorentzian, and Voigt profiles at a representative atmospheric level.

In [ ]:
# Mid-troposphere conditions
T = 250.0    # K
P = 50000.0  # Pa (500 hPa)

dnu_D = doppler_hwhm(nu0, T)
dnu_L = lorentz_hwhm(0.07, P, T)

print(f'At T={T} K, P={P/100:.0f} hPa:')
print(f'  Doppler HWHM  Δν_D = {dnu_D:.5f} cm⁻¹')
print(f'  Lorentz HWHM  Δν_L = {dnu_L:.5f} cm⁻¹')
print(f'  Voigt parameter η  = {dnu_L/(dnu_D+dnu_L):.3f}')
print(f'  → Profile is {"pressure-dominated" if dnu_L > dnu_D else "Doppler-dominated"}')

nu_zoom = np.linspace(6249.8, 6250.2, 2000)
demo_line_profiles(nu_zoom, nu0, dnu_D, dnu_L,
                   savefig='../figures/02b_line_profiles.png')

## 3. Profile Evolution with Altitude

In the upper atmosphere, pressure drops and the profile transitions from Lorentzian to Gaussian.

In [ ]:
# Representative levels from surface to stratosphere
levels = [
    ('Surface',       101325, 288),
    ('700 hPa',        70000, 275),
    ('500 hPa',        50000, 255),
    ('200 hPa',        20000, 220),
    ('Stratosphere',   10000, 218),
]

nu_zoom = np.linspace(6249.7, 6250.3, 3000)
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(levels)))

for (label, P, T), col in zip(levels, colors):
    dnu_D_i = doppler_hwhm(nu0, T)
    dnu_L_i = lorentz_hwhm(0.07, P, T)
    phi_V   = voigt_profile_func(nu_zoom, nu0, dnu_D_i, dnu_L_i)
    ax.plot(nu_zoom, phi_V / phi_V.max(), color=col, lw=1.8, label=f'{label}')

ax.set_xlabel('Wavenumber [cm⁻¹]', fontsize=12)
ax.set_ylabel('Normalised profile', fontsize=12)
ax.set_title('Voigt Profile Evolution with Altitude\n(Lower altitude → broader Lorentzian wings)', fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/02c_profile_altitude.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Multi-Line Cross Section

Real CO₂ spectra consist of many overlapping absorption lines — the absorption cross section is their superposition.

In [ ]:
nu = np.linspace(6220, 6280, 5000)

# Synthetic CO₂ line list (representative values)
line_pos = np.array([6228.35, 6232.78, 6235.44, 6240.55, 6245.60,
                     6251.90, 6254.23, 6259.10, 6262.45])
line_str = np.array([7.9e-24, 1.35e-23, 1.2e-23, 1.55e-23, 8.7e-24,
                     1.25e-23, 8.5e-24, 3.8e-24, 2.4e-24])

dnu_D = doppler_hwhm(6250.0, T)
dnu_L = lorentz_hwhm(0.07, P, T)

xsec_voigt   = absorption_cross_section(nu, line_pos, line_str, dnu_D, dnu_L, 'voigt')
xsec_gauss   = absorption_cross_section(nu, line_pos, line_str, dnu_D, dnu_L, 'gaussian')
xsec_lorentz = absorption_cross_section(nu, line_pos, line_str, dnu_D, dnu_L, 'lorentz')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(nu, xsec_gauss,   '--', color='royalblue', lw=1.2, label='Gaussian', alpha=0.8)
ax.plot(nu, xsec_lorentz, ':',  color='tomato',    lw=1.2, label='Lorentz',  alpha=0.8)
ax.plot(nu, xsec_voigt,         color='seagreen',  lw=2.0, label='Voigt')

# Mark line positions
for nu_i in line_pos:
    ax.axvline(nu_i, color='gray', lw=0.5, ls=':', alpha=0.6)

ax.set_xlabel('Wavenumber [cm⁻¹]', fontsize=12)
ax.set_ylabel('Cross section [cm²/molecule]', fontsize=12)
ax.set_title('CO₂ Absorption Cross Section — Multi-Line Spectrum', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/02d_multiline_xsec.png', dpi=150, bbox_inches='tight')
plt.show()